In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client

In [2]:
client = Client()
client = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_RBF.json")
client.get_next_trials(max_trials=1)

{56: {'n_ci': 1.9968956021445148, 'n_it': 1.2146628525759025}}

In [3]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [4]:
trials = 1000

X_lis = []
for i in range(trials):
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"x1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"x3", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    for i in range(100):
        X = sampler.three.PseudorandomSampler3D_func(27,Parameters_lis).T
        if np.shape(X) != (0,3):
            break
    X_lis.append(X)

y_max_lis = []
for X in X_lis:
    s1_arr = X.T[0]
    s2_arr = X.T[1]
    b1_arr = X.T[2]
    n_ci_lis = []
    for s1,b1 in zip(s1_arr,b1_arr):
        n_ci_lis.append(PredictorsToCaStoichs(s1,b1))
    n_it_lis = []
    for s2,b1 in zip(s2_arr,b1_arr):
        n_it_lis.append(PredictorsToIaStoichs(s2,b1))
    n_ci_arr = np.array(n_ci_lis)
    n_it_arr = np.array(n_it_lis)
    y_pred_lis = []
    for n_ci,n_it in zip(n_ci_arr,n_it_arr):
        y_pred_lis.append(client.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0])
    y_max_lis.append(np.max(np.array(y_pred_lis)))

y_max_arr = np.array(y_max_lis)
print(y_max_arr.tolist())
print(np.average(y_max_arr))

[15.051412190052599, 15.048301639014408, 14.704800089951803, 14.818192644923512, 14.9781545481197, 14.210953947050468, 14.569980542132225, 14.694723026181283, 14.091995227381188, 15.114522073772818, 14.529092580457139, 15.12255912172435, 14.951558808294028, 14.723075306316282, 13.45858579228676, 14.758361753470998, 14.562495848194684, 15.005805366528433, 14.138683067611302, 14.932853132655739, 14.500289943726806, 14.96583886976181, 14.698404069889605, 13.919356331039461, 14.56495830435013, 14.845101545869822, 15.068634101694347, 14.33247756257889, 15.046558506929816, 14.171691041744667, 14.865823850017513, 14.900348793117441, 14.937581592425007, 14.981761765825896, 15.092323928503882, 14.684410467038088, 15.102639337873315, 14.876335077639975, 14.938949414809153, 15.059385437697529, 15.000215832218835, 15.103897003288715, 15.060568433114316, 14.99739936192867, 14.198984316121937, 14.729029181266997, 15.070513787218001, 14.451729230149175, 14.904567632305806, 15.07121848529873, 14.99649

In [5]:
np.average(y_max_arr)

np.float64(14.763252390957096)

In [6]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_RBF/DataGenerated/pseudorandom_27.pkl"
latestdf = pd.DataFrame(y_max_arr)
pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)